# Piping a Prompt, Model and Output Parser

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import CommaSeparatedListOutputParser

In [3]:
list_instructions = CommaSeparatedListOutputParser().get_format_instructions()

In [4]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [10]:
chat_template = ChatPromptTemplate.from_messages({('human', '''I've recently adopted a {pet}, could you suggest three {pet} names? \n''' + list_instructions)})

In [15]:
print(chat_template.messages[0].prompt.template)

I've recently adopted a {pet}, could you suggest three {pet} names? 
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [16]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1, max_completion_tokens=1000)

In [17]:
list_output_parser = CommaSeparatedListOutputParser()

## Old way using individual invocations

In [20]:
chat_template_result = chat_template.invoke({'pet': 'dog'})

In [21]:
chat_template_result

ChatPromptValue(messages=[HumanMessage(content="I've recently adopted a dog, could you suggest three dog names? \nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`", additional_kwargs={}, response_metadata={})])

In [23]:
chat_result = chat.invoke(chat_template_result)

In [24]:
list_output_parser.invoke(chat_result)

['Luna', 'Milo', 'Scout']

## New way using chain object

In [25]:
chain = chat_template | chat | list_output_parser

In [27]:
chain.invoke({'pet': 'dog'})

['Milo', 'Luna', 'Scout']

**This way is much cleaner and we can still use invoke() on each individual object if we need to debug and error.**